> **Note (EPIC #13504)** : ce notebook est désormais une **carte transversale**. Les sections 1--3 détaillées (reconstruction de la borne, témoin extrémal, concentration Hoeffding numérique) ont été absorbées par les sûlings [2.8d](2.8d-Lean-Novikoff-Convergence.ipynb) (Novikoff + témoin) et [2.8b](2.8b-Theorie-PAC-Lean.ipynb) (Hoeffding). Le tableau récapitulatif et la carte des certifications restent ici comme carte de lecture du triptyque.

# 2.8c — Borne + Témoin extrémal + Concentration : ce que `learning_theory_lean` sait déjà faire

**Navigation** : [<< 2.8b-Theorie-PAC-Lean](2.8b-Theorie-PAC-Lean.ipynb) | [Index](../README.md)

**Kernel** : Python 3 (cpu)

**Compagnon formel** : `MyIA.AI.Notebooks/ML/learning_theory_lean/`

---

## Concept

Trois temps, qui distinguent une borne décorative d'une borne **utile** :

```
BORNE          -- une quantite ne peut pas depasser X
TEMOIN EXTR.   -- voici un objet qui atteint X (la borne est serree, pas decorative)
CONCENTRATION  -- voici avec quelle probabilite on s'en ecarte
```

Le triptyque reapparait dans tout le machine learning :
la **borne PAC** est inutile tant qu'on n'a pas son temoin extremal ;
et la **concentration** dit *combien d'echantillons* pour etre proche.

Sur le perceptron de Novikoff, le lake `learning_theory_lean` montre :

- **BORNE** (`Perceptron.Convergence.lean`) : `n <= (R/gamma)^2` mises a jour.
  Deux lemmes : alignement (Lem A) et norme (Lem B), combines par Cauchy-Schwarz.
- **TEMOIN** (`Perceptron.Tightness.lean`) : `witnessPts = [1+I, 1-I]`, separateur `u = 1`,
  `gamma = 1`, `R = sqrt(2)`. Apres 2 mises a jour, `n * gamma^2 = R^2` **exactement**.
- **CONCENTRATION** (`PacLearning.Hoeffding.lean`) : `P[|emp - true| > eps] <= 2 * exp(-2 n eps^2)`.

Ce notebook **consomme** ces theoremes (file:ligne), il ne les re-prouve pas.
Il mesure leur application sur des instances explicites.

In [1]:
import numpy as np
from pathlib import Path
from typing import List, Tuple

RNG = np.random.default_rng(seed=20260822)
print(f'numpy={np.__version__}')


numpy=2.4.2


## Carte des sùblings -- où vit chaque substance ?

Ce notebook est la **carte transversale** du triptyque BORNE / TEMOIN EXTRÉMAL / CONCENTRATION dans `learning_theory_lean/`. Les mesures numériques et les simulations détaillées vivent désormais dans les **sûlings**, chacune étant là où son contenu est canonique :

| Substance | Sûling | Où exactement | Statut |
|---|---|---|---|
| **Reconstruction de la borne `n ≤ (R/γ)²`** (Novikoff jouet Python) | [2.8d-Lean-Novikoff-Convergence](2.8d-Lean-Novikoff-Convergence.ipynb) | section « *La dynamique rejouée sur des entiers* » + exercices 1, 3 | rejoué en Lean (entiers exacts), vérifié sur balayage |
| **Témoin extrémal** `n·γ² = R²` (temps numérique) | [2.8d-Lean-Novikoff-Convergence](2.8d-Lean-Novikoff-Convergence.ipynb) | section « *Le témoin sature la borne* » + `tightnessRun_saturates` | prouvé dans le lake (Tightness.lean:139) |
| **Concentration bilatérale Hoeffding** `P[|emp-true|>eps] ≤ 2exp(-2n eps²)` (Monte-Carlo seedée) | [2.8b-Theorie-PAC-Lean](2.8b-Theorie-PAC-Lean.ipynb) | section 5 « `Hoeffding.lean` » (Monte-Carlo Lean seedée) | rejouée dans le kernel Lean (seed égale) |
| **PAC fini** `n ≥ (log|H|+log(1/δ))/ε` | [2.8b-Theorie-PAC-Lean](2.8b-Theorie-PAC-Lean.ipynb) | section 8 « `PacFiniteBound.lean` » | évaluée par `#eval` (rationnel exact) |

**Pourquoi cette carte reste utile** : un lecteur qui arrive sur le triptyque voit en une table **qui porte quoi**, sans avoir à ouvrir les trois notebooks. La valeur pédagogique n'est pas dans la mesure (les sûlings la font mieux en Lean) mais dans la **mise en regard** des trois theoremes.

> **EPIC #13504** : ce notebook a été réduit à sa **valeur ajoutée propre** (la carte transversale). Les Sections 1, 2, 3 d'origine (mesures numériques Python) ont été absorbées par les sûlings, où elles s'exécutent dans leur kernel canonique (Lean pour Novikoff/Témoin, Lean Hoeffding Monte-Carlo seedée) plutôt qu'en numpy redondant.


## Conclusion -- Le triptyque dans le machine learning

Les **trois temps** que ce notebook a traverses sont recurrents :

| Triptyque | Borne | Temoin extremal | Concentration |
|-----------|-------|-----------------|---------------|
| **Novikoff perceptron** | `n <= (R/gamma)^2` (Convergence.lean) | `n*gamma^2 = R^2` (Tightness.lean) | n/a (algorithme online) |
| **Hoeffding bilateral**  | `P[|emp-true|>eps] <= 2 exp(-2n eps^2)` (Hoeffding.lean) | `bernoulli_subgaussian` (MGF.lean) | convergence en `O(1/sqrt(n))` |
| **PAC fini**             | `n >= (log|H| + log(1/delta)) / eps` (PacFiniteBound.lean) | concept de VC-dim (non formalise ici) | union bound sur `|H|` |

**Le lake comme interprete certifie** : on **consomme** la preuve formelle (le fichier Lean donne
le `file:line` du theoreme), on **mesure** sur des instances explicites, et on **discute** quand
la borne est decorative vs informante.

**Limites du notebook** :

- `lake build SUCCESS` du module `learning_theory_lean` non re-verifie dans ce cycle
  (cache `~/.lake` peut etre orphelin). On s'appuie sur la sortie du compteur de `sorry`
  dans le body PR.
- Le temoin de Hoeffding (`bernoulli_subgaussian`) demande un import `Mathlib.Probability`
  qu'on ne fait pas ici : on **cite** le fichier, on ne le rejoue pas.
- La VC-dimension (generalisation au cas infini) est en dehors du perimetre de ce notebook.

**Suite suggeree** : un notebook 2.8d sur la VC-dimension, qui sort du cas fini pour attaquer
le cas `|H| = infini`. Cela necessiterait probablement un nouveau module Lean ou l'import de
`Mathlib.Probability.Martingale.Basic`.

## Comment vérifier sur main ?

Les vérifications mécaniques étaient historiquement dans ce notebook : existence des théorèmes cités et `sorry = 0` dans le lake. Elles sont désormais portées par les instruments suivants :

- **Existence des théorèmes** : vérifiée par les `#check` explicites dans [2.8d](2.8d-Lean-Novikoff-Convergence.ipynb) (`PerceptronRun.novikoff_mistake_bound`, `tightnessRun_saturates`, ...) et [2.8b](2.8b-Theorie-PAC-Lean.ipynb) (`hoeffding_concentration`, `pac_finite_class_bound`, ...).
- **Zéro `sorry` réel dans le lake** : porté par l'organe `python scripts/lean/count_code_sorry.py --json` (champ `distinct_code_sorry`), mesuré en CI par le job `proof-integrity`. **Pas de `grep -c sorry`** : il sur-compte la prose (docstrings, commentaires).
- **Pas de `native_decide` / `sorryAx` dans le chemin des théorèmes cités** : porté par le job CI `lean-axiom.yml` (catégorie `forbidden`, pas seulement `sorry`).

Pour une **revue complète** du triplet BORNE/TÉMOIN/CONCENTRATION sur le lake courant, suivre les **Voir aussi** internes de chaque sibling.
